# ASD average network atlas — fsaverage5

Visualises the **group-average network atlas** of the ASD cohort (`ds-asd`, task-chase) produced by the *old* fsaverage5 pipeline
(`charles_01/fit_PFM_av01.py`, same approach as `networks_indTopology/`):

- input: group-average CM `correlation_matrices/sub-All/sub-All_ses-1_task-chase_runs-9.npy` (18,715 × 18,715, fsaverage5 without medial wall)
- 30 mm geodesic exclusion → proportional thresholds **3, 4, 5, 10, 20, 40 %** → Infomap per threshold
- each Infomap community relabelled to the **Cole-Anticevic (CA) network** with the highest Jaccard overlap (≥ 0.1, else −1 = unassigned)
- consensus: sparse → dense (thresholds with ≤ 5 networks skipped if < 10 %)

Goal: decide whether (a fsLR-transformed version of) this atlas is a sensible **reference for `04b_relabel.py`** in the new fsLR 32k pipeline (as was done for DNumRisk → `caNets_DDnr`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import hcp_utils as hcp
import nilearn.plotting as nplt
from nilearn.datasets import fetch_surf_fsaverage
from numrisk.fmri_analysis.gradients.utils import get_basic_mask

bids_folder = '/mnt_AdaBD_largefiles/Data/DNumRisk_Data/ds-asd'
nets_folder = f'{bids_folder}/derivatives/networks_infomap_full'
confspec = '36Pscrub3BPfilter'
conn_thresholds = [0.03, 0.04, 0.05, 0.1, 0.2, 0.4]
threshs_str = '-'.join(str(t) for t in conn_thresholds)

ca_names = hcp.ca_network['labels']   # {1: 'Visual1', ..., 12: 'Orbito-Affective'}
mask, _ = get_basic_mask()            # fsaverage5 (20484,), True = cortex w/o medial wall
print(mask.shape, mask.sum())

In [ ]:
all_thresh = np.load(f'{nets_folder}/sub-All_ses-1_task-chase_allThresholds_threshs-{threshs_str}_confspec-{confspec}.npy')
consensus  = np.load(f'{nets_folder}/sub-All_ses-1_task-chase_consensusMapping_threshs-{threshs_str}_confspec-{confspec}.npy')
print(all_thresh.shape, consensus.shape)

def to_fsav5(labels_masked):
    """Masked labels (−1 = unassigned) → full fsaverage5 array (NaN = medial wall, 0 = unassigned)."""
    out = np.full(mask.shape[0], np.nan)
    out[mask] = np.where(labels_masked == -1, 0, labels_masked)
    return out

consensus_fsav5 = to_fsav5(consensus)

## Plotting helper (CA colours, unassigned = grey)

In [ ]:
rgb = np.array(list(hcp.ca_network['rgba'].values())[1:])
cmap_ca = ListedColormap(np.vstack([[0.5, 0.5, 0.5, 1.0], rgb]))   # index 0 = unassigned
fsaverage = fetch_surf_fsaverage('fsaverage5')

def plot_nets_fsav5(labels_fsav5, title='', views=('lateral', 'medial', 'dorsal', 'posterior')):
    """Both hemispheres (rows) × views (columns) on the inflated fsaverage5 surface."""
    fig, axes = plt.subplots(2, len(views), figsize=(4 * len(views), 7),
                             subplot_kw=dict(projection='3d'))
    for i_h, (hemi, lab) in enumerate(zip(['left', 'right'], np.split(labels_fsav5, 2))):
        for i_v, view in enumerate(views):
            nplt.plot_surf(surf_mesh=fsaverage[f'infl_{hemi}'], surf_map=lab,
                           bg_map=fsaverage[f'sulc_{hemi}'], bg_on_data=True, darkness=0.7,
                           hemi=hemi, view=view, cmap=cmap_ca, vmin=0, vmax=12,
                           avg_method='median', colorbar=False, axes=axes[i_h, i_v])
    handles = [plt.Rectangle((0, 0), 1, 1, color=cmap_ca(k)) for k in range(13)]
    fig.legend(handles, ['Unassigned'] + [ca_names[k] for k in range(1, 13)],
               loc='center right', frameon=False)
    fig.suptitle(title, y=0.95)
    fig.subplots_adjust(wspace=0, hspace=0, right=0.85)
    return fig

## Consensus atlas

In [ ]:
fig = plot_nets_fsav5(consensus_fsav5, title='ASD sub-All — consensus (CA-relabelled)')

## Network sizes per threshold

Vertex counts per CA network for each threshold and the consensus. Missing = network not present at that threshold.

In [ ]:
rows = {f'{t*100:g}%': pd.Series(r).value_counts() for t, r in zip(conn_thresholds, all_thresh)}
rows['consensus'] = pd.Series(consensus).value_counts()
sizes = pd.DataFrame(rows).fillna(0).astype(int).sort_index()
sizes.index = ['Unassigned' if k == -1 else ca_names[k] for k in sizes.index]
sizes

## Individual thresholds

In [ ]:
for t, r in zip(conn_thresholds, all_thresh):
    plot_nets_fsav5(to_fsav5(r), title=f'ASD sub-All — threshold {t*100:g}%')
    plt.show()

## Notes / observations

- Thresholds here (3–40 % density) are much **denser** than in the new fsLR pipeline (Gordon 2017: 0.3–5 %). At 40 % only 4 CA networks survive.
- **Dorsal-attention (5)** only appears at the sparsest threshold (3 %) — every DAN vertex in the consensus comes from that single level, which makes the DAN (our network of interest) fragile in this atlas.
- **Visual2 (2)** is nearly absent from the consensus; Somatomotor / Default are large.
- Labels are *already* Jaccard-assigned to the CA atlas, so this map is partly CA-atlas-driven, not a purely data-driven ASD atlas.

TODO: transform to fsLR 32k (as done for DNumRisk `caNets_DDnr`) and compare with `gordon17` / `caNets_DDnr` as candidate references for `nets_PFM/04b_relabel.py`.